In [120]:
# imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, classification_report

In [121]:
# load data
train_df_raw = pd.read_csv("../data/raw/train.csv")
test_df_raw = pd.read_csv("../data/raw/test.csv")

In [122]:
# clean data
#make it so I can run this cell over and over again
train_df = train_df_raw
test_df = test_df_raw


## encode sex
sex_map = {
    "male":0,
    "female":1
}
train_df["Sex"] = train_df["Sex"].replace(sex_map)

#encode Embarked
train_df = pd.get_dummies(train_df, columns=["Embarked"], dtype = int)
train_df

#drop cabin number and ticket number
train_df = train_df.drop(columns=["Cabin", "Ticket"])

# fill fare/age with median
train_df["Age"] = train_df["Age"].fillna(train_df["Age"].median())
train_df["Fare"] = train_df["Fare"].fillna(train_df["Fare"].median())

#save before I drop name
train_df_title = train_df.copy()

#drop name cause its a string
train_df = train_df.drop(columns=["Name"])

#save ids
train_ids = train_df["PassengerId"]
test_ids = test_df["PassengerId"]

# drop ids
train_df = train_df.drop(columns=["PassengerId"])
test_df = test_df.drop(columns=["PassengerId"])

# test set changes
test_df["Sex"] = test_df["Sex"].replace(sex_map)
test_df = pd.get_dummies(test_df, columns=["Embarked"], dtype = int)
test_df = test_df.drop(columns=["Cabin", "Ticket"])
test_df["Age"] = test_df["Age"].fillna(test_df["Age"].median())
test_df["Fare"] = test_df["Fare"].fillna(test_df["Fare"].median())
test_df = test_df.drop(columns=["Name"])

train_df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
0,0,3,0,22.0,1,0,7.2500,0,0,1
1,1,1,1,38.0,1,0,71.2833,1,0,0
2,1,3,1,26.0,0,0,7.9250,0,0,1
3,1,1,1,35.0,1,0,53.1000,0,0,1
4,0,3,0,35.0,0,0,8.0500,0,0,1
...,...,...,...,...,...,...,...,...,...,...
886,0,2,0,27.0,0,0,13.0000,0,0,1
887,1,1,1,19.0,0,0,30.0000,0,0,1
888,0,3,1,28.0,1,2,23.4500,0,0,1
889,1,1,0,26.0,0,0,30.0000,1,0,0


In [123]:
#create my model
X = train_df.drop(columns = ["Survived"])
y = train_df["Survived"]

#split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# get predictions
predictions = model.predict(X_val)

In [124]:
#inspect model
accuracy = accuracy_score(y_val, predictions)
print(accuracy)

print(confusion_matrix(y_val, predictions))
print(classification_report(y_val, predictions))

0.8100558659217877
[[90 15]
 [19 55]]
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



In [125]:
# combine family columns and add one for the oerson themself. Feature engineering
train_df_fam = train_df.copy()
train_df_fam["family_size"] = train_df_fam["SibSp"] + train_df_fam["Parch"] + 1
# train_df_fam = train_df_fam.drop(columns=["SibSp", "Parch"])
train_df_fam

#create my model
X_fam = train_df_fam.drop(columns = ["Survived"])
y_fam = train_df_fam["Survived"]

#split
X_train_fam, X_val_fam, y_train_fam, y_val_fam = train_test_split(
    X_fam,
    y_fam,
    test_size=0.2,
    random_state=42
)

#train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_fam, y_train_fam)

# get predictions
fam_predictions = model.predict(X_val_fam)

In [126]:
#inspect fam model
accuracy_fam = accuracy_score(y_val_fam, fam_predictions)
print(accuracy_fam)

print(confusion_matrix(y_val_fam, fam_predictions))
print(classification_report(y_val_fam, fam_predictions))

0.8100558659217877
[[90 15]
 [19 55]]
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



In [127]:
# title extraction
train_df_title["Title"] = train_df_title["Name"].str.extract(r",\s*([^\.]+)\.")
train_df_title = train_df_title.drop(columns=["PassengerId", "Name"])

#title grouping
misc_titles = [
    "Major",
    "Mlle",
    "Col",
    "Don",
    "Mme",
    "Ms",
    "Lady",
    "Sir",
    "Capt",
    "the Countess",
    "Jonkheer",
    "Rev",
    "Dr"
]
train_df_title["Title"] = train_df_title["Title"].replace(misc_titles, 'Misc')

print(train_df_title["Title"].value_counts())

#encode titles
train_df_title = pd.get_dummies(train_df_title, columns=["Title"], dtype = int)

train_df_title

Title
Mr        517
Miss      182
Mrs       125
Master     40
Misc       27
Name: count, dtype: int64


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Misc,Title_Miss,Title_Mr,Title_Mrs
0,0,3,0,22.0,1,0,7.2500,0,0,1,0,0,0,1,0
1,1,1,1,38.0,1,0,71.2833,1,0,0,0,0,0,0,1
2,1,3,1,26.0,0,0,7.9250,0,0,1,0,0,1,0,0
3,1,1,1,35.0,1,0,53.1000,0,0,1,0,0,0,0,1
4,0,3,0,35.0,0,0,8.0500,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,0,27.0,0,0,13.0000,0,0,1,0,1,0,0,0
887,1,1,1,19.0,0,0,30.0000,0,0,1,0,0,1,0,0
888,0,3,1,28.0,1,2,23.4500,0,0,1,0,0,1,0,0
889,1,1,0,26.0,0,0,30.0000,1,0,0,0,0,0,1,0


In [128]:
#create my model
X_title = train_df_title.drop(columns = ["Survived"])
y_title = train_df_title["Survived"]

#split
X_train_title, X_val_title, y_train_title, y_val_title = train_test_split(
    X_title,
    y_title,
    test_size=0.2,
    random_state=42
)

#train model
model_title = LogisticRegression(max_iter=1000)
model_title.fit(X_train_title, y_train_title)

# get predictions
title_predictions = model_title.predict(X_val_title)

In [129]:
#inspect title model
accuracy_title = accuracy_score(y_val_title, title_predictions)
print(accuracy_title)

print(confusion_matrix(y_val_title, title_predictions))
print(classification_report(y_val_title, title_predictions))

0.8100558659217877
[[88 17]
 [17 57]]
              precision    recall  f1-score   support

           0       0.84      0.84      0.84       105
           1       0.77      0.77      0.77        74

    accuracy                           0.81       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



In [130]:
# save dfs
train_df.to_csv("../data/processed/train_clean.csv", index=False)
test_df.to_csv("../data/processed/test_clean.csv", index=False)

Random Forests

In [131]:
#random forests
from sklearn.ensemble import RandomForestClassifier

In [132]:
#create forest
forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=5
)

#train forest
forest.fit(X_train, y_train)

#get predictions
forest_predictions = forest.predict(X_val)

#evaluate
print(accuracy_score(y_val, forest_predictions))
print(confusion_matrix(y_val, forest_predictions))
print(classification_report(y_val, forest_predictions))

0.8156424581005587
[[95 10]
 [23 51]]
              precision    recall  f1-score   support

           0       0.81      0.90      0.85       105
           1       0.84      0.69      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.80       179
weighted avg       0.82      0.82      0.81       179



In [133]:
print(forest.score(X_train, y_train))
print(forest.score(X_val, y_val))

0.8581460674157303
0.8156424581005587


In [134]:
importance = pd.Series(
    forest.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importance)

Sex           0.482937
Pclass        0.143707
Fare          0.135404
Age           0.120250
SibSp         0.043059
Parch         0.033945
Embarked_S    0.019386
Embarked_C    0.012188
Embarked_Q    0.009125
dtype: float64


In [139]:
# add is child
train_df_child = train_df.copy()

train_df_child["IsChild"] = (
    train_df_child['Age'] < 16
).astype(int)

#create my model
X_child = train_df_child.drop(columns = ["Survived"])
y_child = train_df_child["Survived"]

#split
X_train_child, X_val_child, y_train_child, y_val_child = train_test_split(
    X_child,
    y_child,
    test_size=0.2,
    random_state=42
)

forest_child = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=5
)

#train forest
forest_child.fit(X_train_child, y_train_child)

#get predictions
forest_predictions_child = forest_child.predict(X_val_child)

#evaluate
print(accuracy_score(y_val_child, forest_predictions_child))
print(confusion_matrix(y_val_child, forest_predictions_child))
print(classification_report(y_val_child, forest_predictions_child))


0.8212290502793296
[[96  9]
 [23 51]]
              precision    recall  f1-score   support

           0       0.81      0.91      0.86       105
           1       0.85      0.69      0.76        74

    accuracy                           0.82       179
   macro avg       0.83      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179



In [140]:
print(forest_child.score(X_train_child, y_train_child))
print(forest_child.score(X_val_child, y_val_child))

0.8595505617977528
0.8212290502793296


In [141]:
#importance
importance = pd.Series(
    forest_child.feature_importances_,
    index=X_train_child.columns
).sort_values(ascending=False)

print(importance)

Sex           0.447399
Fare          0.160995
Pclass        0.136161
Age           0.105207
SibSp         0.058319
Parch         0.031373
IsChild       0.021669
Embarked_S    0.018232
Embarked_C    0.012999
Embarked_Q    0.007646
dtype: float64


Add old age indicator

In [154]:
# add old age
train_df_old = train_df_child.copy()

train_df_old["IsOld"] = (
    train_df_child['Age'] > 60
    ).astype(int)

#create my model
X_old = train_df_old.drop(columns = ["Survived"])
y_old = train_df_old["Survived"]

#split
X_train_old, X_val_old, y_train_old, y_val_old = train_test_split(
    X_old,
    y_old,
    test_size=0.2,
    random_state=42
)

forest_old = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    max_depth=5
)

#train forest
forest_old.fit(X_train_old, y_train_old)

#get predictions
forest_predictions_old = forest_old.predict(X_val_old)

#evaluate
print(accuracy_score(y_val_old, forest_predictions_old))
print(confusion_matrix(y_val_old, forest_predictions_old))
print(classification_report(y_val_old, forest_predictions_old))

0.8212290502793296
[[95 10]
 [22 52]]
              precision    recall  f1-score   support

           0       0.81      0.90      0.86       105
           1       0.84      0.70      0.76        74

    accuracy                           0.82       179
   macro avg       0.83      0.80      0.81       179
weighted avg       0.82      0.82      0.82       179



In [155]:
print(forest_old.score(X_train_old, y_train_old))
print(forest_old.score(X_val_old, y_val_old))

0.8497191011235955
0.8212290502793296


In [156]:
#importance
importance = pd.Series(
    forest_old.feature_importances_,
    index=X_train_old.columns
).sort_values(ascending=False)

print(importance)

Sex           0.382533
Fare          0.184002
Pclass        0.152980
Age           0.099885
SibSp         0.056182
Parch         0.035163
IsChild       0.031998
Embarked_C    0.021967
Embarked_S    0.020804
Embarked_Q    0.012054
IsOld         0.002432
dtype: float64
